# Task 2 — Qwen Dense Model Training (MaxText)

This notebook runs **Qwen3-0.6B (base)** and a **scaled-up ~1B variant** for 50 steps each, on **CPU**, **GPU**, and **TPU** Colab runtimes.

### How to use this notebook
1. Set **Runtime > Change runtime type** to `CPU`, run all cells, download `results_cpu/`.
2. Switch to `T4 GPU`, restart runtime, run all cells again, download `results_gpu/`.
3. Switch to `TPU v2-8` (or whatever TPU is offered), restart runtime, run all cells again, download `results_tpu/`.
4. Copy all three `results_*` folders into the repo's `results/` directory.

Each run is fully self-contained (re-clones MaxText, re-installs deps) so it works independently of which backend Colab gives you.

## 0. Detect backend

In [ ]:
import subprocess, os, sys

def detect_backend():
    try:
        subprocess.run(['nvidia-smi'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return 'gpu'
    except Exception:
        pass
    if 'COLAB_TPU_ADDR' in os.environ or 'TPU_NAME' in os.environ:
        return 'tpu'
    return 'cpu'

BACKEND = detect_backend()
print(f'Detected backend: {BACKEND}')

## 1. Install MaxText (backend-specific)

In [ ]:
!git clone -q https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

if BACKEND == 'tpu':
    !pip install -q -e ".[tpu]"
elif BACKEND == 'gpu':
    !pip install -q -e ".[gpu]"
else:
    !pip install -q -e "."

print('Install complete for backend:', BACKEND)

In [ ]:
# Sanity check: confirm JAX sees the expected devices
import jax
print('JAX devices:', jax.devices())
print('JAX backend:', jax.default_backend())

## 2. Write configs

We inline the configs here (mirrors `configs/qwen/*.yml` in the repo) so the notebook is fully standalone.

In [ ]:
qwen_06b_overrides = dict(
    model_name='qwen3-0.6b',
    steps=50,
    dataset_type='synthetic',
    per_device_batch_size=1,
    max_target_length=512,
    learning_rate=3e-4,
    dtype='bfloat16',
    weight_dtype='bfloat16',
    enable_checkpointing=False,
    attention='dot_product',
    tokenizer_type='huggingface',
    tokenizer_path='Qwen/Qwen3-0.6B',
    log_period=1,
)

# Scaled-up ~1B: wider emb_dim + mlp_dim, same Qwen3 block, fewer layers
# See configs/qwen/qwen3_1b_train.yml for full justification.
qwen_1b_overrides = dict(
    model_name='qwen3-0.6b',
    base_emb_dim=1536,
    base_num_query_heads=16,
    base_num_kv_heads=8,
    base_mlp_dim=4096,
    base_num_decoder_layers=24,
    head_dim=96,
    steps=50,
    dataset_type='synthetic',
    per_device_batch_size=1,
    max_target_length=512,
    learning_rate=3e-4,
    dtype='bfloat16',
    weight_dtype='bfloat16',
    enable_checkpointing=False,
    attention='dot_product',
    tokenizer_type='huggingface',
    tokenizer_path='Qwen/Qwen3-0.6B',
    log_period=1,
)

print('Configs ready.')

## 3. Helper to run training and capture full logs

In [ ]:
import subprocess, time, os, json, re

RESULTS_DIR = f'/content/results_{BACKEND}'
os.makedirs(RESULTS_DIR, exist_ok=True)

def run_training(run_name, overrides, log_filename):
    args = [f'{k}={v}' for k, v in overrides.items()]
    args.append(f'base_output_directory=/content/maxtext_output')
    args.append(f'run_name={run_name}')

    cmd = ['python', '-m', 'maxtext.train', 'src/maxtext/configs/base.yml'] + args
    print('Running:', ' '.join(cmd))

    log_path = os.path.join(RESULTS_DIR, log_filename)
    start = time.time()
    with open(log_path, 'w') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='')
            f.write(line)
        proc.wait()
    elapsed = time.time() - start
    print(f'\n=== Finished {run_name} in {elapsed:.1f}s, exit code {proc.returncode} ===')
    return log_path, elapsed, proc.returncode

## 4. Run A — Qwen 0.6B base

In [ ]:
log_path_06b, wall_time_06b, exit_code_06b = run_training(
    run_name=f'qwen_0.6b_{BACKEND}',
    overrides=qwen_06b_overrides,
    log_filename=f'qwen_0.6b_{BACKEND}_log.txt'
)

## 5. Run B — Qwen ~1B scaled-up

In [ ]:
log_path_1b, wall_time_1b, exit_code_1b = run_training(
    run_name=f'qwen_1b_{BACKEND}',
    overrides=qwen_1b_overrides,
    log_filename=f'qwen_1b_{BACKEND}_log.txt'
)

## 6. Parse metrics out of the logs

MaxText logs per-step metrics to stdout in a parseable format and also writes a `metrics.json`-style trace. We extract every numeric field reported (loss, step time, TFLOPS/s, tokens/s, learning rate, gradient norm, memory) — **no metric is selectively dropped.**

In [ ]:
import re, pandas as pd

METRIC_PATTERNS = {
    'step': r'completed step:\s*(\d+)',
    'loss': r'loss:\s*([\d.]+)',
    'step_time_s': r'step_time_seconds:\s*([\d.]+)|seconds:\s*([\d.]+)',
    'tflops_per_sec': r'TFLOP/s/device:\s*([\d.]+)|tflops_per_sec:\s*([\d.]+)',
    'tokens_per_sec': r'tokens/s/device:\s*([\d.]+)|tokens_per_second:\s*([\d.]+)',
    'learning_rate': r'learning[_/]rate:\s*([\d.eE+-]+)',
    'grad_norm': r'grad[_ ]norm:\s*([\d.]+)',
    'param_norm': r'param[_ ]norm:\s*([\d.]+)',
    'perplexity': r'perplexity:\s*([\d.]+)',
    'memory_gb': r'memory.*?:\s*([\d.]+)\s*GB',
}

def parse_log(path):
    rows = []
    with open(path) as f:
        text = f.read()
    # Split roughly by step-completion lines; fallback: scan line by line and accumulate
    lines = text.splitlines()
    current = {}
    for line in lines:
        for key, pattern in METRIC_PATTERNS.items():
            m = re.search(pattern, line)
            if m:
                val = next((g for g in m.groups() if g is not None), None)
                if val is not None:
                    current[key] = float(val) if key != 'step' else int(val)
        if 'completed step' in line and current:
            rows.append(current.copy())
    return pd.DataFrame(rows)

df_06b = parse_log(log_path_06b)
df_1b = parse_log(log_path_1b)

print('Qwen 0.6B parsed rows:', len(df_06b))
print('Qwen 1B parsed rows:', len(df_1b))
df_06b.tail()

In [ ]:
# Save parsed metrics as CSV for the comparison table in analysis/final_summary.md
df_06b.to_csv(os.path.join(RESULTS_DIR, f'qwen_0.6b_{BACKEND}_metrics.csv'), index=False)
df_1b.to_csv(os.path.join(RESULTS_DIR, f'qwen_1b_{BACKEND}_metrics.csv'), index=False)

summary = {
    'backend': BACKEND,
    'qwen_0.6b': {
        'wall_time_total_s': wall_time_06b,
        'exit_code': exit_code_06b,
        'avg_step_time_s': df_06b['step_time_s'].mean() if 'step_time_s' in df_06b else None,
        'final_loss': df_06b['loss'].iloc[-1] if 'loss' in df_06b and len(df_06b) else None,
        'avg_tflops_per_sec': df_06b['tflops_per_sec'].mean() if 'tflops_per_sec' in df_06b else None,
        'avg_tokens_per_sec': df_06b['tokens_per_sec'].mean() if 'tokens_per_sec' in df_06b else None,
    },
    'qwen_1b': {
        'wall_time_total_s': wall_time_1b,
        'exit_code': exit_code_1b,
        'avg_step_time_s': df_1b['step_time_s'].mean() if 'step_time_s' in df_1b else None,
        'final_loss': df_1b['loss'].iloc[-1] if 'loss' in df_1b and len(df_1b) else None,
        'avg_tflops_per_sec': df_1b['tflops_per_sec'].mean() if 'tflops_per_sec' in df_1b else None,
        'avg_tokens_per_sec': df_1b['tokens_per_sec'].mean() if 'tokens_per_sec' in df_1b else None,
    }
}

with open(os.path.join(RESULTS_DIR, f'summary_{BACKEND}.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

## 7. Download results

Run this cell, download the zip, then unzip it into `results/qwen_0_6b/` and `results/qwen_1b/` in the repo (matched to this backend).

In [ ]:
import shutil
from google.colab import files

zip_path = f'/content/results_{BACKEND}'
shutil.make_archive(zip_path, 'zip', RESULTS_DIR)
files.download(zip_path + '.zip')

---
### Next step

Re-run this entire notebook after switching **Runtime > Change runtime type** to the next backend (CPU → GPU → TPU, in any order), so you accumulate `results_cpu/`, `results_gpu/`, `results_tpu/` for both Qwen 0.6B and Qwen 1B.